<a href="https://colab.research.google.com/github/toryor31oct/group15-Fun-Rai-Khwam-plod-Phai/blob/Tang-oh/fun_rai_kwam_plod_phai(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# โหลดข้อมูลจากไฟล์ customer.csv และ policy.csv

In [3]:
import random
import sqlite3
import pandas as pd

df_customer = pd.read_csv("customer.csv")
df_policy = pd.read_csv("policy.csv")

# รวมข้อมูลลูกค้าและกรมธรรม์เพื่อเตรียมใช้จำลองการเคลม
df_base_policies = pd.merge(
    df_policy,
    df_customer[[
        "Customer_ID",
        "Customer_Name",
        "Beneficiary_Name",
        "Beneficiary_Relationship",
    ]],
    on="Customer_ID",
    how="inner",
)

# Class จากการวิเคราะห์องค์กร 3 Classes

In [4]:
# Class 1: เก็บข้อมูลคำร้องขอเคลมประกันภัยแต่ละรายการ
class ClaimRequest:
  def __init__(
      self, claim_id, policy_id, customer_id, insurance_type, claim_amount
  ):
    self.claim_id = claim_id
    self.policy_id = policy_id
    self.customer_id = customer_id
    self.insurance_type = insurance_type  # ประกันชีวิต, ประกันสุขภาพ, ประกันอุบัติเหตุ, ประกันโรคร้ายแรง
    self.claim_amount = claim_amount
    self.status = "Pending"  # ทุกคำร้องเริ่มที่สถานะ Pending

# Class 2: ประเมินอนุมัติเคลมตามกฎเงื่อนไขของประเภทประกัน
class ClaimAssessment:
  def __init__(self, assessment_id, claim_id):
    self.assessment_id = assessment_id
    self.claim_id = claim_id
    self.approval_status = "Pending"
    self.assessment_note = ""

  def process_approval(self, claim_amount, coverage_limit, insurance_type):
    """Method ตัดสินใจอนุมัติเคลมตามกฎซับซ้อนของแต่ละประเภทประกัน"""
    if insurance_type == "ประกันชีวิต":
      # ประกันชีวิต: อนุมัติจ่ายเต็มวงเงินคุ้มครอง (ทุนประกัน) กรณีมรณกรรมแก่ผู้รับประโยชน์
      self.approval_status = "Approved"
      self.assessment_note = (
          "อนุมัติจ่ายตามทุนประกันชีวิตเต็มจำนวนกรณีมรณกรรมแก่ผู้รับประโยชน์"
      )

    elif insurance_type == "ประกันโรคร้ายแรง":
      # ประกันโรคร้ายแรง: อนุมัติจ่ายเงินก้อน (Lump Sum) เมื่อตรวจพบโรคร้ายแรงตามเงื่อนไข
      self.approval_status = "Approved"
      self.assessment_note = (
          "อนุมัติจ่ายเงินก้อน (Lump Sum) เมื่อตรวจพบโรคร้ายแรงระยะที่คุ้มครอง"
      )

    elif insurance_type in ["ประกันสุขภาพ", "ประกันอุบัติเหตุ"]:
      # ประกันสุขภาพ / อุบัติเหตุ: อนุมัติตามค่ารักษาจริง แต่ต้องไม่เกินวงเงินคุ้มครอง
      if claim_amount <= coverage_limit:
        self.approval_status = "Approved"
        self.assessment_note = (
            "อนุมัติตามค่ารักษาพยาบาลจริง (อยู่ในวงเงินคุ้มครอง)"
        )
      else:
        self.approval_status = "Rejected"
        self.assessment_note = (
            f"ปฏิเสธ: ยอดขอเบิก ({claim_amount:,.2f} บาท) เกินวงเงินคุ้มครอง"
            f" ({coverage_limit:,.2f} บาท)"
        )

    return self.approval_status

# Class 3: จัดการคำนวณและสั่งจ่ายค่าสินไหมทดแทน
class ClaimPayment:
  def __init__(self, payment_id, claim_id):
    self.payment_id = payment_id
    self.claim_id = claim_id
    self.paid_amount = 0.0
    self.payment_status = "Unpaid"

  def process_payment(
      self, claim_amount, coverage_limit, insurance_type, approval_status
  ):
    """Method คำนวณยอดเงินจ่ายจริงตามผลการประเมิน"""
    if approval_status == "Approved":
      if insurance_type in ["ประกันชีวิต", "ประกันโรคร้ายแรง"]:
        # ประกันชีวิต และ โรคร้ายแรง จ่ายเต็มทุนประกันภัย
        self.paid_amount = coverage_limit
      else:
        # ประกันสุขภาพ และ อุบัติเหตุ จ่ายตามยอดขอเบิกจริง
        self.paid_amount = claim_amount
      self.payment_status = "Paid"
    else:
      self.paid_amount = 0.0
      self.payment_status = "Cancelled"
    return self.payment_status



# Helper Functions

In [5]:
def generate_claim_details(insurance_type):
  """ฟังก์ชันที่ 1: สุ่มรายละเอียดสาเหตุการยื่นเคลมจำแนกตามประเภทประกัน"""
  details_map = {
      "ประกันชีวิต": [
          "เสียชีวิตจากโรคเจ็บป่วยรุนแรง",
          "เสียชีวิตจากอุบัติเหตุทางถนน",
          "เสียชีวิตจากภาวะหัวใจล้มเหลว",
      ],
      "ประกันสุขภาพ": [
          "เข้ารับการรักษาผู้ป่วยใน (IPD) โรคไข้หวัดใหญ่",
          "ผ่าตัดไส้ติ่งอักเสบฉุกเฉิน",
          "เข้ารับการรักษาผู้ป่วยนอก (OPD) ลำไส้อักเสบ",
      ],
      "ประกันอุบัติเหตุ": [
          "เข้ารักษาฉุกเฉินจากลื่นล้มหกล้ม",
          "กระดูกข้อเท้าแตกหักจากการเล่นกีฬา",
          "แผลฉกรรจ์จากอุบัติเหตุในบ้านเรือน",
      ],
      "ประกันโรคร้ายแรง": [
          "ตรวจพบโรคมะเร็งระยะเริ่มต้น/ระยะรุนแรง",
          "ตรวจพบโรคหลอดเลือดสมองแตกหรือตีบ",
          "ป่วยด้วยโรคกล้ามเนื้อหัวใจตายจากการขาดเลือด",
      ],
  }
  options = details_map.get(insurance_type, ["การเคลมค่าสินไหมทั่วไป"])
  return random.choice(options)


def generate_claim_amount(coverage_limit):
  """ฟังก์ชันที่ 2: สุ่มยอดเงินขอเบิกค่ารักษา/สินไหม"""
  return round(random.uniform(10000.0, coverage_limit * 1.15), 2)


def calculate_net_payout(paid_amount, tax_rate=0.01):
  """ฟังก์ชันที่ 3: คำนวณเงินสุทธิหลังหักภาษี ณ ที่จ่าย (มี default argument: tax_rate=0.01)"""
  return round(paid_amount * (1 - tax_rate), 2)


# จำลองข้อมูลทีละรายการด้วย Loop (300 รายการ)

In [6]:
random.seed(42)  # ล็อค seed สำหรับ debug
requests_list = []
assessments_list = []
payments_list = []

for i in range(len(df_base_policies)):
  policy_row = df_base_policies.iloc[i]

  c_id = policy_row["Customer_ID"]
  p_id = policy_row["Policy_ID"]
  ins_type = policy_row["Plan_Name"]
  coverage = float(policy_row["Coverage_Amount"])

  clm_id = f"CLM{i+1:03d}"
  clm_amount = generate_claim_amount(coverage)

  # 1. สร้าง Object คำร้องยื่นเคลม
  req = ClaimRequest(clm_id, p_id, c_id, ins_type, clm_amount)
  claim_detail = generate_claim_details(ins_type)

  # 2. สร้าง Object ประเมินอนุมัติ
  ass_id = f"ASS{i+1:03d}"
  ass = ClaimAssessment(ass_id, req.claim_id)
  app_status = ass.process_approval(req.claim_amount, coverage, ins_type)
  req.status = app_status

  # 3. สร้าง Object สั่งจ่ายเงินสินไหม
  pay_id = f"PAY{i+1:03d}"
  pay = ClaimPayment(pay_id, req.claim_id)
  pay_status = pay.process_payment(
      req.claim_amount, coverage, ins_type, ass.approval_status
  )

  # เก็บ Object และข้อมูลที่เกี่ยวข้อง
  requests_list.append((req, policy_row, claim_detail))
  assessments_list.append(ass)
  payments_list.append(pay)

print(
    f"จำลองการรับและประมวลผลเคลมสำเร็จแล้วทั้งหมด {len(requests_list)} รายการ"
)


จำลองการรับและประมวลผลเคลมสำเร็จแล้วทั้งหมด 300 รายการ


# แปลง List ของ Object เป็น DataFrame และบันทึกเป็น CSV

In [7]:
# ใช้ List Comprehension ดึง Attribute และเรียก Method ของแต่ละ Object
records = [{
    "claim_id": req.claim_id,
    "customer_id": req.customer_id,
    "customer_name": row["Customer_Name"],
    "beneficiary_name": row["Beneficiary_Name"],
    "beneficiary_relationship": row["Beneficiary_Relationship"],
    "policy_id": req.policy_id,
    "insurance_type": req.insurance_type,
    "claim_detail": detail,
    "coverage_limit": float(row["Coverage_Amount"]),
    "claim_amount": req.claim_amount,
    "assessment_id": ass.assessment_id,
    "approval_status": ass.approval_status,
    "assessment_note": ass.assessment_note,
    "payment_id": pay.payment_id,
    "paid_amount": pay.paid_amount,
    "net_payout": calculate_net_payout(pay.paid_amount),
    "payment_status": pay.payment_status,
} for (req, row, detail), ass, pay in zip(
    requests_list, assessments_list, payments_list
)]

df_claims = pd.DataFrame(records)
csv_filename = "claims_system_300.csv"
df_claims.to_csv(csv_filename, index=False, encoding="utf-8-sig")
print(f"บันทึกไฟล์ CSV สำเร็จ ขนาด: {df_claims.shape}")


บันทึกไฟล์ CSV สำเร็จ ขนาด: (300, 17)


# สร้างฐานข้อมูล SQLite แยก 3 ตาราง

In [8]:
conn = sqlite3.connect("claims_system.db")

# บันทึกแยกตารางเพื่อทำ Relational Database
df_claims[[
    "claim_id",
    "customer_id",
    "policy_id",
    "insurance_type",
    "claim_detail",
    "coverage_limit",
    "claim_amount",
]].to_sql("claim_requests", conn, if_exists="replace", index=False)

df_claims[[
    "assessment_id",
    "claim_id",
    "approval_status",
    "assessment_note",
]].to_sql("claim_assessments", conn, if_exists="replace", index=False)

df_claims[[
    "payment_id",
    "claim_id",
    "paid_amount",
    "net_payout",
    "payment_status",
]].to_sql("claim_payments", conn, if_exists="replace", index=False)


conn.close()
display(df_claims.head(300))

,claim_id,customer_id,customer_name,beneficiary_name,beneficiary_relationship,policy_id,insurance_type,claim_detail,coverage_limit,claim_amount,assessment_id,approval_status,assessment_note,payment_id,paid_amount,net_payout,payment_status
0,CLM001,C001,ธนภัทร บุญมี,รัตนาภรณ์ แสงทอง,สามี,POL0001,ประกันชีวิต,เสียชีวิตจากโรคเจ็บป่วยรุนแรง,550000.0,408043.18,ASS001,Approved,อนุมัติจ่ายตามทุนประกันชีวิตเต็มจำนวนกรณีมรณกร...,PAY001,550000.00,544500.00,Paid
1,CLM002,C002,ภัทรพล เจริญสุข,ศิริพร มีสุข,พี่น้อง,POL0002,ประกันสุขภาพ,เข้ารับการรักษาผู้ป่วยใน (IPD) โรคไข้หวัดใหญ่,400000.0,343697.72,ASS002,Approved,อนุมัติตามค่ารักษาพยาบาลจริง (อยู่ในวงเงินคุ้ม...,PAY002,343697.72,340260.74,Paid
2,CLM003,C003,ศุภชัย สุวรรณ,วรพล พรหมดี,พี่น้อง,POL0003,ประกันชีวิต,เสียชีวิตจากภาวะหัวใจล้มเหลว,2350000.0,610994.91,ASS003,Approved,อนุมัติจ่ายตามทุนประกันชีวิตเต็มจำนวนกรณีมรณกร...,PAY003,2350000.00,2326500.00,Paid
3,CLM004,C004,ธีรภัทร์ รุ่งเรือง,ศุภชัย พรหมดี,บิดา,POL0004,ประกันชีวิต,เสียชีวิตจากภาวะหัวใจล้มเหลว,750000.0,97377.14,ASS004,Approved,อนุมัติจ่ายตามทุนประกันชีวิตเต็มจำนวนกรณีมรณกร...,PAY004,750000.00,742500.00,Paid
4,CLM005,C005,กิตติพงษ์ สุวรรณ,เอกภพ อินทร์ทอง,บิดา,POL0005,ประกันชีวิต,เสียชีวิตจากโรคเจ็บป่วยรุนแรง,2250000.0,2309592.84,ASS005,Approved,อนุมัติจ่ายตามทุนประกันชีวิตเต็มจำนวนกรณีมรณกร...,PAY005,2250000.00,2227500.00,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,CLM296,C296,มณีรัตน์ อินทร์ทอง,ณิชาภัทร พรหมดี,ภรรยา,POL0296,ประกันสุขภาพ,ผ่าตัดไส้ติ่งอักเสบฉุกเฉิน,1200000.0,741749.37,ASS296,Approved,อนุมัติตามค่ารักษาพยาบาลจริง (อยู่ในวงเงินคุ้ม...,PAY296,741749.37,734331.88,Paid
296,CLM297,C297,ธนภัทร ทองคำ,กัญญารัตน์ สุขสวัสดิ์,ภรรยา,POL0297,ประกันชีวิต,เสียชีวิตจากภาวะหัวใจล้มเหลว,650000.0,446520.92,ASS297,Approved,อนุมัติจ่ายตามทุนประกันชีวิตเต็มจำนวนกรณีมรณกร...,PAY297,650000.00,643500.00,Paid
297,CLM298,C298,ชยพล สุวรรณ,อรปรียา เจริญสุข,มารดา,POL0298,ประกันสุขภาพ,เข้ารับการรักษาผู้ป่วยนอก (OPD) ลำไส้อักเสบ,1450000.0,54921.64,ASS298,Approved,อนุมัติตามค่ารักษาพยาบาลจริง (อยู่ในวงเงินคุ้ม...,PAY298,54921.64,54372.42,Paid
298,CLM299,C299,ภัทรพล อินทร์ทอง,กิตติพงษ์ สุขสวัสดิ์,ภรรยา,POL0299,ประกันชีวิต,เสียชีวิตจากอุบัติเหตุทางถนน,2900000.0,1435137.46,ASS299,Approved,อนุมัติจ่ายตามทุนประกันชีวิตเต็มจำนวนกรณีมรณกร...,PAY299,2900000.00,2871000.00,Paid
